# Estadísticas descriptivas e histogramas (EPH)
**Práctica Profesional I - Rita Josefina Juan Briones**

Este cuaderno calcula las **estadísticas descriptivas univariadas** (una variable por vez) y hace **histogramas** para ver cómo se distribuyen los datos.

Recordá: ejecutá las celdas **en orden**, con ▶ o **Shift + Enter**. Lo que va después de `#` es un comentario.

## Paso 1: Cargar los datos
El archivo `eph_tecnologia_microdatos.csv` tiene que estar subido en Colab (carpeta 📁 de la izquierda). Si no está, la celda te avisa.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt   # libreria para hacer graficos

try:
    df = pd.read_csv("eph_tecnologia_microdatos.csv")
    print("Archivo cargado:", df.shape[0], "filas y", df.shape[1], "columnas")
except FileNotFoundError:
    print("No encuentro el archivo. Subilo desde la carpeta de la izquierda y volve a ejecutar.")

## Paso 2: Estadísticas descriptivas de las variables numéricas
Las variables numéricas clave son:
- `IX_TOT`: cantidad de integrantes del hogar
- `ITF`: ingreso total familiar (en pesos)

Calculamos 5 medidas con `agg()`:
- **media**: el promedio
- **mediana**: el valor del medio (la mitad de los hogares está por debajo y la otra mitad por encima)
- **mínimo** y **máximo**: el valor más bajo y el más alto
- **desvío estándar**: qué tan dispersos están los datos alrededor de la media

pandas **ignora los datos faltantes (NaN)** automáticamente en estos cálculos.

In [ ]:
variables_clave = ["IX_TOT", "ITF"]

# agg() calcula varias medidas juntas; .T da vuelta la tabla para que se lea mejor
tabla = df[variables_clave].agg(["mean", "median", "min", "max", "std"]).T
tabla.columns = ["Media", "Mediana", "Minimo", "Maximo", "Desvio estandar"]
tabla = tabla.round(2)
tabla

In [ ]:
# Cuantos datos validos (no nulos) tiene cada variable
df[variables_clave].count()

## Paso 3: Interpretar los resultados
Dos cosas para mirar:
1. **Media vs mediana.** Si la media es bastante mayor que la mediana, hay pocos valores muy altos que "tiran" el promedio hacia arriba (distribución asimétrica a la derecha).
2. **Coeficiente de variación** = desvío / media. Sirve para comparar la dispersión entre variables que tienen unidades distintas. Cuanto más alto, más dispersa la variable.

In [ ]:
# Coeficiente de variacion de cada variable
cv = (tabla["Desvio estandar"] / tabla["Media"]).round(2)
print("Coeficiente de variacion:")
print(cv)

# Diferencia entre media y mediana del ingreso
dif = tabla.loc["ITF", "Media"] - tabla.loc["ITF", "Mediana"]
print("\nEn ITF la media supera a la mediana en $", round(dif, 2))

## Paso 4 (extra): Percentiles
Los **percentiles** dividen los datos ordenados. El 25 % deja abajo al 25 % de los hogares; el 50 % es la mediana; el 75 % deja abajo al 75 %.

In [ ]:
df[variables_clave].quantile([0.25, 0.50, 0.75]).round(0)

## Paso 5: Variables de categorías (tiene PC / tiene internet)
Estas variables no tienen "media": se describen con **porcentajes**.

Primero hay que sacar los **9 (sin respuesta)**: los convertimos en `NaN` en una **copia** (`df_cat`) para no tocar los datos originales. Esta no es la limpieza completa, es lo mínimo para poder calcular porcentajes correctos.

In [ ]:
df_cat = df.copy()   # copia: el df original queda intacto
df_cat[["NIVEL_ED", "IV12_3", "II7"]] = df_cat[["NIVEL_ED", "IV12_3", "II7"]].replace(9, np.nan)

# normalize=True da proporciones; multiplicamos por 100 para tener porcentajes
pc = (df_cat["IV12_3"].value_counts(normalize=True) * 100).round(1)
internet = (df_cat["II7"].value_counts(normalize=True) * 100).round(1)

print("Tiene computadora (1 = Si, 2 = No):")
print(pc)
print("\nTiene internet fija (1 = Si, 2 = No):")
print(internet)

## Paso 6: Histograma del ingreso (ITF)
Un **histograma** muestra cómo se distribuyen los valores de una variable numérica: divide el rango en tramos iguales llamados **bins** (las "barras") y cuenta cuántos hogares caen en cada tramo.

- El eje horizontal (x) es el ingreso; el vertical (y) es la cantidad de hogares.
- Con `plt.hist()` se hace. Es obligatorio sacar los NaN antes con `dropna()`, si no da error.
- Agregamos líneas verticales para la **media** y la **mediana** con `plt.axvline()`.

In [ ]:
itf = df["ITF"].dropna()   # sacamos los nulos

plt.figure(figsize=(9, 4.5))
plt.hist(itf, bins=30, color="steelblue", edgecolor="white")
plt.axvline(itf.mean(), color="red", linestyle="--", label=f"Media: ${itf.mean():,.0f}")
plt.axvline(itf.median(), color="orange", linestyle="-", label=f"Mediana: ${itf.median():,.0f}")
plt.title("Distribución del Ingreso Total Familiar (ITF)")
plt.xlabel("Ingreso (pesos)")
plt.ylabel("Cantidad de hogares")
plt.legend()
plt.show()

## Paso 7: Histograma de la cantidad de integrantes
`IX_TOT` toma valores enteros (1, 2, 3...). Para que cada barra sea un número exacto, definimos los bins a mano: `np.arange(0.5, 10.5, 1)` crea cortes en 0,5 / 1,5 / 2,5..., así cada entero queda en el centro de su barra.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["IX_TOT"], bins=np.arange(0.5, 10.5, 1), color="seagreen", edgecolor="white")
plt.title("Cantidad de integrantes por hogar")
plt.xlabel("Integrantes")
plt.ylabel("Cantidad de hogares")
plt.xticks(range(1, 10))
plt.show()

## Paso 8: Cómo cambia el histograma según los bins
La cantidad de bins cambia mucho cómo se ve. Pocos bins esconden detalles; demasiados hacen un gráfico "ruidoso". Probá estos tres:

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(14, 3.8))   # 1 fila, 3 graficos
for eje, b in zip(ejes, [5, 30, 100]):
    eje.hist(itf, bins=b, color="steelblue", edgecolor="white")
    eje.set_title(f"{b} bins")
    eje.set_xlabel("Ingreso (pesos)")
ejes[0].set_ylabel("Cantidad de hogares")
plt.tight_layout()
plt.show()

## Cómo leer un histograma
- **Forma:** si es simétrico (campana) o tiene una "cola" larga hacia un lado (asimétrico).
- **Dónde se concentran** la mayoría de los datos.
- **Valores atípicos:** barras aisladas muy lejos del resto.

En el ingreso (ITF) vas a ver que la mayoría de los hogares está a la izquierda y hay una cola larga hacia la derecha: pocos hogares con ingresos muy altos. Por eso la media queda por encima de la mediana.

**Resultados esperados:** IX_TOT media 3,31 / mediana 3 / desvío 1,48. ITF media ≈ 773.680 / mediana ≈ 604.132 / desvío ≈ 576.672. Tiene PC: 63,9 % / Tiene internet fija: 73,5 %.